<a href="https://colab.research.google.com/github/steventeng2022/ml-colab-1/blob/main/Perceptron_MLP_Activation_Backpropagation_%E6%95%99%E6%9D%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 深度學習基礎教材：Perceptron → MLP → Activation → Backpropagation

**適用程度：** 具備 Python、NumPy 基本知識，開始學 PyTorch 的讀者。  
**學習成果：** 能解釋 XOR 限制、推導 Chain Rule、用 NumPy 手刻梯度、利用 PyTorch autograd 訓練 MLP。  
**建議學習時間：** 4–6 小時（含練習）。

> 本教材為可直接執行的 Jupyter Notebook，章節中的程式碼可由上至下執行。練習題含 TODO 與 Hint。

## 0. 環境準備與路線圖

從 **單一神經元** 開始，先學 **加權和 + 偏置 + 激活函數**，接著串成 MLP，最後用梯度下降與反向傳播訓練。

依賴：`numpy`, `matplotlib`，最後兩節額外需要 `torch`。若未安裝 PyTorch，可先完成前七節。

```bash
pip install numpy matplotlib torch
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)

## 1. Perceptron：單一神經元

輸入向量 $x\in\mathbb{R}^d$、權重 $w\in\mathbb{R}^d$、偏置 $b\in\mathbb{R}$：

$$z=w^Tx+b=\sum_{i=1}^{d}w_ix_i+b$$

原始 Perceptron 採用階躍函數：$\hat y=\mathbb{1}[z\geq0]$。

- **Weight** 控制不同特徵的重要性（及正負方向）。
- **Bias** 讓決策邊界不必通過原點。
- **Decision boundary** 為 $w^Tx+b=0$；二維時是一條直線。
- **原始 Perceptron 更新**：當樣本分類有誤，$w\leftarrow w+\eta(y-\hat y)x$，$b\leftarrow b+\eta(y-\hat y)$。

注意：Perceptron 更新規則與後續使用可微 Loss 的梯度下降不是同一種演算法。

In [ ]:
def step(z):
    return (np.asarray(z) >= 0).astype(int)

def perceptron(x, w, b):
    return step(np.dot(x, w) + b)

x = np.array([1.0, 0.0])
w = np.array([1.0, 1.0])
print('prediction:', perceptron(x, w, -0.5))

# 真值表：AND
X_logic = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
print('AND:', perceptron(X_logic, np.array([1.,1.]), -1.5))

### 1.1 用資料訓練原始 Perceptron

以下用 AND 的四筆資料示範逐樣本更新；如果資料線性可分，Perceptron 在適當條件下可收斂。

In [ ]:
def train_perceptron(X, y, lr=0.1, epochs=20):
    w = np.zeros(X.shape[1], dtype=float)
    b = 0.0
    history = []
    for _ in range(epochs):
        errors = 0
        for xi, yi in zip(X, y):
            pred = int(np.dot(xi, w) + b >= 0)
            delta = lr * (yi - pred)
            w += delta * xi
            b += delta
            errors += (pred != yi)
        history.append(errors)
        if errors == 0:
            break
    return w, b, history

y_and = np.array([0,0,0,1])
w_and, b_and, hist_and = train_perceptron(X_logic, y_and)
print('weights:', w_and, 'bias:', b_and, 'errors per epoch:', hist_and)
print('predictions:', perceptron(X_logic, w_and, b_and))

## 2. Linear Separability 與 XOR 問題

XOR 的四筆資料是 `(0,0)→0`, `(0,1)→1`, `(1,0)→1`, `(1,1)→0`。正類位於對角，無法由**單條直線**分開，所以單一 Perceptron 無法完全正確分類。

**重要延伸：** 不是「任何沒有 Hidden Layer 的模型都不能解 XOR」；如果先人工設計非線性特徵，線性分類器也可能在新特徵空間分開。這正是 MLP 自動學習非線性表示的價值。

In [ ]:
y_xor = np.array([0,1,1,0])
fig, ax = plt.subplots(figsize=(5,4))
for label, marker in [(0, 'o'), (1, 's')]:
    pts = X_logic[y_xor == label]
    ax.scatter(pts[:,0], pts[:,1], s=130, marker=marker, label=f'XOR = {label}')
ax.set(xlim=(-0.35,1.35), ylim=(-0.35,1.35), xlabel='x1', ylabel='x2', title='XOR is not linearly separable')
ax.legend(); ax.grid(alpha=.2); plt.show()

w_xor, b_xor, hist_xor = train_perceptron(X_logic, y_xor, epochs=30)
print('Single perceptron predictions:', perceptron(X_logic, w_xor, b_xor))

## 3. Multi-Layer Perceptron（MLP）

以矩陣（每一列代表一筆樣本）表示：

$$Z^{(1)}=XW^{(1)}+b^{(1)},\quad A^{(1)}=g(Z^{(1)})$$
$$Z^{(2)}=A^{(1)}W^{(2)}+b^{(2)},\quad \hat Y=\sigma(Z^{(2)})$$

例如 batch size = 4、2 個輸入特徵、4 個隱藏神經元、1 個輸出：

| 變數 | Shape |
|---|---|
| $X$ | `(4, 2)` |
| $W^{(1)}$ | `(2, 4)` |
| $b^{(1)}$ | `(4,)` |
| $A^{(1)}$ | `(4, 4)` |
| $W^{(2)}$ | `(4, 1)` |
| $\hat Y$ | `(4, 1)` |

**參數數量：** 第一層 $2\times4+4=12$；第二層 $4\times1+1=5$；總計 **17 個可訓練參數**。

### 為什麼 Activation 必須非線性？

若沒有非線性函數，$W_2(W_1x+b_1)+b_2$ 可合併成單一仿射變換；增加層數仍然沒有學會非線性決策邊界。

In [ ]:
rng = np.random.default_rng(7)
W1 = rng.normal(size=(2,4)); b1 = np.zeros((1,4))
W2 = rng.normal(size=(4,1)); b2 = np.zeros((1,1))
relu = lambda z: np.maximum(0,z)
sigmoid = lambda z: 1/(1+np.exp(-np.clip(z,-60,60)))
Z1 = X_logic @ W1 + b1
A1 = relu(Z1)
Z2 = A1 @ W2 + b2
Yhat = sigmoid(Z2)
for name, val in [('X',X_logic),('Z1',Z1),('A1',A1),('Z2',Z2),('Yhat',Yhat)]:
    print(name, val.shape)
print('predicted probabilities:', Yhat.ravel())

## 4. Activation Function 及其導數

| 函數 | 公式 | 導數/特性 | 常見用途 |
|---|---|---|---|
| Step | $\mathbb 1[z\ge0]$ | 幾乎處處導數為 0 | 原始感知器 |
| Sigmoid | $1/(1+e^{-z})$ | $\sigma(z)(1-\sigma(z))$；大幅度輸入易飽和 | 二元分類輸出 |
| Tanh | $\tanh(z)$ | $1-\tanh^2(z)$；輸出中心在 0 | 某些循環網路 |
| ReLU | $\max(0,z)$ | 正區域為 1、負區域為 0 | 常見隱藏層 |
| Leaky ReLU | $\max(\alpha z,z)$ | 負區域斜率為 $\alpha$ | 緩解 dying ReLU |
| GELU | $z\Phi(z)$ | 平滑的非線性門控 | Transformer/現代架構 |

**梯度消失：** Sigmoid/Tanh 在飽和區域的導數接近 0，經多層 Chain Rule 相乘可能讓前面層的梯度非常小。**Dying ReLU：** 若某神經元對訓練樣本持續輸出負的 pre-activation，梯度可能長期為 0。ReLU 在 $z=0$ 不可微，框架會指定實務使用的次梯度（PyTorch 為 0）。

In [ ]:
z = np.linspace(-5, 5, 500)
functions = {
    'Sigmoid': sigmoid(z), 'Tanh': np.tanh(z),
    'ReLU': np.maximum(0,z), 'Leaky ReLU': np.maximum(.1*z,z),
}
fig, axes = plt.subplots(1,2,figsize=(11,4))
for name, val in functions.items():
    axes[0].plot(z, val, label=name)
    axes[1].plot(z, np.gradient(val,z), label=name)
axes[0].set_title('Activation functions'); axes[1].set_title('Approximate derivatives')
for ax in axes:
    ax.set(xlabel='z'); ax.grid(alpha=.2); ax.legend()
plt.tight_layout(); plt.show()

## 5. Loss Function 與 Gradient Descent

**Regression：** 均方誤差 $MSE=\frac1N\sum_i(\hat y_i-y_i)^2$。  
**Binary Classification：** $BCE=-\frac1N\sum_i\{y_i\log p_i+(1-y_i)\log(1-p_i)\}$。

$\theta\leftarrow\theta-\eta\nabla_\theta L$，$\eta$ 是 learning rate。太大可能震盪或發散，太小可能收斂緩慢。

**實務技巧：** 在 PyTorch 中，二元分類偏好 `BCEWithLogitsLoss` 搭配模型的 **raw logits**，內部數值處理比手動 `sigmoid` 再 `BCELoss` 更穩定。

## 6. Backpropagation：從最簡單的 Chain Rule 開始

單一神經元：

$$z=wx+b,\ a=\sigma(z),\ L=\tfrac12(a-y)^2$$

計算圖：$w\to z\to a\to L$。其梯度：

$$\frac{\partial L}{\partial w}=\frac{\partial L}{\partial a}\frac{\partial a}{\partial z}\frac{\partial z}{\partial w}=(a-y)a(1-a)x$$

$$\frac{\partial L}{\partial b}=(a-y)a(1-a)$$

**注意：** 上述簡單形式對應的是 **$\tfrac12$ 平方誤差**，不是 BCE。對 sigmoid + BCE，若對單筆樣本求導，$\partial L/\partial z=a-y$，因為微分中的項會約掉。

### 兩層 MLP 的矩陣反向傳播

設 $N$ 筆樣本、隱藏層 $A_1=g(Z_1)$、輸出層 sigmoid 與**平均 BCE**：

$$dZ_2=(\hat Y-Y)/N$$
$$dW_2=A_1^TdZ_2,\quad db_2=\sum_{\mathrm{batch}}dZ_2$$
$$dA_1=dZ_2W_2^T$$
$$dZ_1=dA_1\odot g'(Z_1)$$
$$dW_1=X^TdZ_1,\quad db_1=\sum_{\mathrm{batch}}dZ_1$$

矩陣相乘的次序反映了 Chain Rule。$\odot$ 代表逐元素相乘。這些式子也顯示了 **Backward 使用 Forward 階段的中間量**。

In [ ]:
# 手算單一權重的梯度，並對照有限差分
x, w, b, y = 2.0, 0.4, -0.1, 1.0
loss_fn = lambda ww: 0.5 * (float(sigmoid(ww*x+b))-y)**2
a = float(sigmoid(w*x+b))
grad_analytic = (a-y)*a*(1-a)*x
eps = 1e-5
grad_numeric = (loss_fn(w+eps)-loss_fn(w-eps))/(2*eps)
print('analytic:', grad_analytic, 'numerical:', grad_numeric)
print('absolute difference:', abs(grad_analytic-grad_numeric))

## 7. NumPy 手刻兩層 MLP 解 XOR

為了清楚觀察梯度，我們使用 **2 → 4 → 1** 的網路、隱藏層 `tanh`、輸出 `sigmoid`、平均 BCE，採全批次梯度下降。種子固定以利重現。

In [ ]:
class NumpyMLP:
    def __init__(self, hidden=4, seed=1):
        r = np.random.default_rng(seed)
        self.W1 = r.normal(0, 0.8, size=(2,hidden))
        self.b1 = np.zeros((1,hidden))
        self.W2 = r.normal(0, 0.8, size=(hidden,1))
        self.b2 = np.zeros((1,1))

    def forward(self, X):
        self.X = X
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = np.tanh(self.Z1)
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.P = sigmoid(self.Z2)
        return self.P

    def backward(self, Y):
        N = self.X.shape[0]
        dZ2 = (self.P - Y) / N    # sigmoid + mean BCE
        dW2 = self.A1.T @ dZ2
        db2 = dZ2.sum(axis=0, keepdims=True)
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * (1-self.A1**2)
        dW1 = self.X.T @ dZ1
        db1 = dZ1.sum(axis=0, keepdims=True)
        return dict(W1=dW1, b1=db1, W2=dW2, b2=db2)

    def step(self, gradients, lr):
        for name, grad in gradients.items():
            setattr(self, name, getattr(self, name) - lr * grad)

def bce_numpy(p, y):
    p = np.clip(p,1e-10,1-1e-10)
    return -np.mean(y*np.log(p)+(1-y)*np.log(1-p))

X = X_logic.copy()
Y = y_xor.reshape(-1,1).astype(float)
model = NumpyMLP(hidden=4, seed=1)
losses = []
for epoch in range(12000):
    p = model.forward(X)
    losses.append(bce_numpy(p,Y))
    grads = model.backward(Y)
    model.step(grads, lr=0.5)

p = model.forward(X)
print('XOR probabilities:', p.ravel())
print('XOR classes:', (p.ravel()>=0.5).astype(int))
print('target:', Y.ravel().astype(int))
plt.plot(losses); plt.title('NumPy MLP training loss'); plt.xlabel('Epoch'); plt.ylabel('BCE'); plt.grid(alpha=.2); plt.show()

### 7.1 梯度檢查（Gradient Checking）

對一個隨機參數使用中央差分：

$$\frac{\partial L}{\partial\theta}\approx\frac{L(\theta+\epsilon)-L(\theta-\epsilon)}{2\epsilon}$$

這可以檢驗手刻 Backward 是否有維度、轉置或縮放錯誤；有限差分是近似值，$b5$ 過大或過小都可能造成誤差。

In [ ]:
check_model = NumpyMLP(hidden=4, seed=8)
check_model.forward(X)
grad = check_model.backward(Y)['W1'][0,0]
epsilon = 1e-5
original = check_model.W1[0,0]
check_model.W1[0,0] = original + epsilon
plus = bce_numpy(check_model.forward(X),Y)
check_model.W1[0,0] = original - epsilon
minus = bce_numpy(check_model.forward(X),Y)
check_model.W1[0,0] = original
numerical = (plus-minus)/(2*epsilon)
print('dW1[0,0] analytic:', grad)
print('dW1[0,0] numeric :', numerical)
print('abs difference   :', abs(grad-numerical))
assert np.isclose(grad,numerical,atol=1e-6)

## 8. PyTorch：autograd 與 nn.Module

PyTorch 的 `autograd` 會記錄支援梯度的運算，`loss.backward()` 從純量 Loss 出發計算梯度並累加至 `parameter.grad`。

典型訓練順序：

1. `optimizer.zero_grad()` 清除上一輪累積的梯度。
2. `logits = model(X)` 執行 Forward。
3. `loss = criterion(logits, y)` 計算 Loss。
4. `loss.backward()` 計算 Backward。
5. `optimizer.step()` 更新參數。

若缺少 PyTorch，以下儲存格會印出提示而不阻斷前面的 NumPy 教學。

In [ ]:
try:
    import torch
    from torch import nn
    print('PyTorch version:', torch.__version__)
except ImportError:
    torch = None
    print('尚未安裝 torch：可先完成 NumPy 章節，再 pip install torch')

In [ ]:
if torch is not None:
    torch.manual_seed(4)
    X_t = torch.tensor(X_logic, dtype=torch.float32)
    y_t = torch.tensor(y_xor.reshape(-1,1), dtype=torch.float32)

    class XORNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(2, 8),
                nn.Tanh(),
                nn.Linear(8, 1),  # raw logits; sigmoid goes into evaluation only
            )
        def forward(self, x):
            return self.net(x)

    net = XORNet()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.05)
    history = []
    for epoch in range(1500):
        optimizer.zero_grad()
        logits = net(X_t)
        loss = criterion(logits, y_t)
        loss.backward()
        optimizer.step()
        history.append(loss.item())

    with torch.no_grad():
        probs = torch.sigmoid(net(X_t))
        print('probabilities:', probs.squeeze().numpy())
        print('class predictions:', (probs.squeeze() >= 0.5).int().numpy())
    plt.plot(history); plt.title('PyTorch XOR training loss'); plt.xlabel('Epoch'); plt.ylabel('BCEWithLogitsLoss'); plt.show()

### 8.1 觀察梯度與 Chain Rule

`requires_grad=True` 告訴 PyTorch 追蹤運算；`.backward()` 會沿計算圖反向套用局部導數。`.grad` 是累積的，不會在每次反向傳播前自動清零。

In [ ]:
if torch is not None:
    w = torch.tensor(0.4, requires_grad=True)
    x = torch.tensor(2.0)
    b = torch.tensor(-0.1)
    y = torch.tensor(1.0)
    a = torch.sigmoid(w*x+b)
    L = 0.5*(a-y)**2
    L.backward()
    print('autograd dL/dw:',w.grad.item())
    print('manual dL/dw:',(a.item()-1)*a.item()*(1-a.item())*2)

## 9. TODO 練習（先自己做）

**練習 A｜Perceptron（基礎）**  
實作 OR 的權重與偏置，使四筆輸入預測為 `[0,1,1,1]`。  
Hint：`w=[1,1]` 時，想想 Bias 必須落在哪個範圍。

**練習 B｜Shape（基礎）**  
輸入 `(32,10)`，兩個 Hidden Layer 分別 16、8 個神經元，輸出 3 類，寫出每個 `W, b, A` Shape 並計算參數總量。  
Hint：`nn.Linear(in_features, out_features)` 的參數數量 = `in_features * out_features + out_features`。

**練習 C｜微分（核心）**  
自行計算 $z=wx+b$、$a=\sigma(z)$、$L=\tfrac12(a-y)^2$ 的 $dL/dw$ 與 $dL/db$。  
Hint：先寫 $dL/da$、$da/dz$、$dz/dw$。

**練習 D｜手刻（進階）**  
將 NumPy MLP 隱藏層 `tanh` 改為 `ReLU`，同時修改 Backward。比較是否成功學會 XOR。  
Hint：`dZ1 = dA1 * (Z1 > 0)`；不同初始種子、學習率會影響訓練。

**練習 E｜PyTorch（進階）**  
把網路改為 `2 → 8 → 8 → 1`，比較 `Tanh` 和 `ReLU` 在 XOR 上的訓練 loss。  
Hint：每次實驗重新建立模型和 optimizer；固定 seed，才方便比較。

In [ ]:
# TODO A: OR Perceptron
# w_or = np.array([..., ...], dtype=float)
# b_or = ...
# prediction = perceptron(X_logic, w_or, b_or)
# assert np.array_equal(prediction, np.array([0,1,1,1]))

# TODO B: 計算 10 -> 16 -> 8 -> 3 的參數數量
# parameter_count = ...
# print(parameter_count)

# TODO C: 手算 sigmoid + half-squared error
# def manual_grads(x, w, b, y):
#     ...
#     return dw, db

# TODO D: 在前面的 NumPyMLP 中，把 tanh 換為 ReLU，並修改導數
# TODO E: 使用 nn.Sequential 擴充隱藏層

## 10. 理解測驗（先作答再展開答案）

**Q1.** 為什麼兩層線性變換不等於學會非線性？  
**Q2.** ReLU 在負半軸的導數通常是多少？  
**Q3.** Backpropagation 與 Gradient Descent 是同一件事嗎？  
**Q4.** `optimizer.zero_grad()` 的目的？  
**Q5.** 為什麼 `BCEWithLogitsLoss` 不應先手動套 `sigmoid`？  
**Q6.** $X:(64,5)$、$W:(5,10)$、$b:(10,)$，輸出 shape 是多少？

<details><summary>點我展開參考答案與解析</summary>

1. 仿射變換合成後仍然是仿射變換，不能自動表達非線性邊界。
2. 0；正半軸為 1，原點的實作慣例可不同。
3. 不同：Backprop 計算梯度，Gradient Descent 使用梯度更新權重。
4. 清除上次累積的梯度，避免不預期的梯度相加。
5. 因為它已經在內部結合 sigmoid 與 BCE 並做穩定計算，應傳入 logits。
6. `(64,10)`；b 透過 broadcasting 加入每筆樣本。

**練習 B 參數總數：** $(10\times16+16)+(16\times8+8)+(8\times3+3)=339$。

</details>

## 11. 延伸學習方向

接下來建議循序學：**Softmax 與 CrossEntropyLoss → Mini-batch 與 SGD/Adam → Initialization → Vanishing/Exploding Gradients → Dropout、Batch Normalization → CNN 與 Transformer**。

**自我檢核：** 你應能不看文件寫出 `nn.Linear` 的 Shape、手推兩層 MLP 的 $dW_1$ / $dW_2$，以及解釋為什麼 XOR 是多層網路的經典入門案例。